# Reproducing Mah et al. (2023) from DANDI:001169: behaviour

Paper: Mah, Schiereck, Bossio and Constantinople (2023), "Distinct value computations support rapid sequential decisions", Nature Communications 14, 7573, doi:10.1038/s41467-023-43250-x.

Dataset: DANDI:001169, version 0.261120.1450 (doi:10.48324/dandi.001169/0.261120.1450). This version was asked for by the prompt of this repeat attempt; it is the most recent published version (published 2026-11-20). The paper cites no dandiset version: its Data availability names the Zenodo deposit doi:10.5281/zenodo.10031483, which this notebook reads for Figure 4 only (all final-stage sessions, which the dandiset does not hold).

Claims file: `claims/mah-2023.yaml`, written from the version of record (Europe PMC, PMC10663503) with Supplementary Information, peer review file and reporting summary, and the paper's code (github.com/constantinoplelab/published, rat_behavior, at b7b68ca8c107). Method details the claims file lacks are cited to the paper or that code where they are used.

The NWB files are streamed: each file is fetched over HTTPS into memory and read with h5py; nothing is written to disk.

Claims, and where they are tested (all 33 are tested; none is blocked):

| Claim | Notebook | Data |
|---|---|---|
| cohort-size, fig1e, fig1f, fig1i, suppfig2, suppfig3, suppfig12, suppfig14 | this one | dandiset |
| fig2a, suppfig5, suppfig6, fig2c-d, fig2e, fig2f, fig2g-h, fig2j, fig2k, fig2l | this one | dandiset |
| fig3c, bayes-observer-offer-reduces-uncertainty | this one | dandiset |
| pre-cue-session-link, fig5b, fig5c, fig5d | this one | dandiset, with rat names and dates from Figure5.m |
| fig4a, fig4c, fig4d | this one | the paper's Zenodo deposit (A_Structs_all) |
| fig3j, suppfig8, fig3f, suppfig9, fig3i, fig3l | `models_mah_2023.ipynb` | dandiset |

## Data check

The cell below prints what was loaded and checks the description against the values.

In [ ]:
import zlib, time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import mah2023 as M

def rng_for(claim, k=0, offset=0):
    """The claim's generator (protocol: seed = crc32 of '<paper key>/<claim id>'), plus `offset`
    for the seed-sensitivity runs (+1000 ... +4000) and k for a second generator."""
    return np.random.default_rng(zlib.crc32(f"mah-2023/{claim}".encode()) + k + offset)

T0 = time.time()
R = {}  # numbers quoted in the summary

In [ ]:
info = M.version_info()
assets = M.list_assets()
sessions, trials = M.load_all(assets)
print(f"loaded {len(sessions)} sessions, {len(trials)} trials in {time.time()-T0:.0f} s")

In [ ]:
trials = M.add_measures(trials)
t = trials
rats = np.sort(t["subject"].unique())
sex = sessions.drop_duplicates("subject").set_index("subject")["sex"]
print("peak memory (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

In [ ]:
print("Dandiset", M.DANDISET, "version", M.VERSION, "| assets:", len(assets),
      "| summary subjects:", info["metadata"]["assetsSummary"]["numberOfSubjects"])
print("sessions:", len(sessions), "| rats:", sessions.subject.nunique(), "| trials:", len(t))
print("sex (subject metadata):", sex.value_counts().to_dict())
print("protocol names:", sessions.protocol.value_counts().to_dict())
print("training_stage values:", t.training_stage.value_counts().to_dict(), "(9 = the paper's stage 8)")
print("block codes (1 mixed, 2 high, 3 low):", t.block.value_counts().sort_index().to_dict())
print("offered volumes (uL):", t.reward_volume_ul.value_counts().sort_index().to_dict())
print("volumes by sex:", t.groupby(t.subject.map(sex)).reward_volume_ul.unique().map(sorted).to_dict())
print("catch_percentage values:", t.catch_percentage.value_counts().sort_index().to_dict())
print("processed rows equal raw trial rows in every session:", bool((sessions.n_trials == sessions.n_raw_trials).all()))
print("trials per session: min", sessions.n_trials.min(), "median", sessions.n_trials.median(), "max", sessions.n_trials.max())
print("session dates:", sessions.start.min(), "to", sessions.start.max())
print("center_port_cue task argument:", sessions.center_port_cue.value_counts().to_dict())
ct = pd.crosstab(t.is_catch, t.is_opt_out); print("catch x opt-out\n", ct)
print("rewarded x violation\n", pd.crosstab(t.is_rewarded, t.is_violation))
for c in ["wait_time", "wait_time_unthresholded", "wait_time_threshold", "reward_delay",
          "inter_trial_interval", "wait_for_center_poke", "wt", "iti"]:
    v = t[c]
    print(f"{c:24s} n={v.notna().sum():9d} min={v.min():9.3f} median={v.median():8.3f} max={v.max():9.3f} (s)")
# reward delay of catch trials, and wait time vs threshold
print("reward_delay on is_catch trials:", t.loc[t.is_catch, "reward_delay"].round(3).value_counts().head(3).to_dict())
over = (t.wait_time > t.wait_time_threshold + 1e-9).sum()
print("wait_time above the rat's wait_time_threshold:", int(over))
thr = t.groupby("subject").wait_time_threshold.nunique()
print("rats with one threshold over all sessions:", int((thr == 1).sum()), "of", len(thr))
# the threshold is the pooled mean + 1 SD of catch-trial wait times (description); recompute it
dep = t.groupby("subject").wait_time_threshold.first()
for name, m in [("opt-out catch trials", t.is_catch & t.is_opt_out),
                ("all catch trials with a wait time (timed-out ones included)", t.is_catch & t.wait_time_unthresholded.notna())]:
    cw = t.loc[m].groupby("subject").wait_time_unthresholded.agg(["mean", "std"])
    rec = cw["mean"] + cw["std"]
    print(f"threshold recomputed from {name}: median |diff| from deposited =",
          round(float((rec - dep).abs().median()), 3), "s; max", round(float((rec - dep).abs().max()), 3))
to = t.is_catch & ~t.is_opt_out & ~t.is_violation
print("catch trials neither opt-out nor violation (timed out):", int(to.sum()),
      "| their unthresholded wait time median", t.loc[to, "wait_time_unthresholded"].median())
hit = t.is_rewarded
print("hit trials: wait_time equals reward_delay in", round(float(np.isclose(t.loc[hit, 'wait_time_unthresholded'], t.loc[hit, 'reward_delay'], atol=1e-3).mean()), 4))
# duplicates: identical trial sequences deposited twice
sig = t.groupby("sess").apply(lambda g: hash((len(g), tuple(np.round(g.start_time.to_numpy()[:50], 3)), tuple(g.reward_volume_ul.to_numpy()[:50]))))
dup = sig[sig.duplicated(keep=False)]
print("sessions whose first 50 trial start times and offers equal another session's:", len(dup))
if len(dup):
    print(sessions.loc[dup.index, ["path"]].assign(sig=dup.values).sort_values("sig").to_string())
# continuity of the rig's trial counter (trials_in_stage) across a rat's deposited sessions
gap = sessions.trials_in_stage_first.to_numpy()[1:] - sessions.trials_in_stage_last.to_numpy()[:-1]
same = sessions.subject.to_numpy()[1:] == sessions.subject.to_numpy()[:-1]
g = gap[same]
print("trials_in_stage at each rat's first deposited session: median",
      sessions.groupby("subject").trials_in_stage_first.first().median(),
      "| between consecutive deposited sessions, counter gap > 5 trials in", int((g > 5).sum()), "of", int(same.sum()),
      f"pairs; rats with any such gap: {int(pd.Series(g > 5).groupby(sessions.subject.to_numpy()[1:][same]).any().sum())}")

## cohort-size

> The dataset comes from 291 rats, with a median of 33,493 behavioral trials and 71 sessions per rat.

Counted over the deposited sessions (all stage 8, the code's stage 9; see the data check for why these are the included sessions).

In [ ]:
n_rats = t.subject.nunique()
tr_per_rat = t.groupby("subject").size()
se_per_rat = sessions.groupby("subject").size()
R["cohort"] = (n_rats, tr_per_rat.median(), se_per_rat.median())
print("rats:", n_rats, "(expected 291, tolerance 0)")
print("median trials per rat:", tr_per_rat.median(), "(expected 33493 +- 3349); range", tr_per_rat.min(), "-", tr_per_rat.max())
print("median sessions per rat:", se_per_rat.median(), "(expected 71 +- 7); range", se_per_rat.min(), "-", se_per_rat.max())

**Result: reproduced.** The dandiset holds 291 rats with a median of 33,493 trials and 71 sessions per rat, the paper's numbers exactly.

## fig1e-wait-time-increases-with-reward

> In mixed blocks, rats wait longer for larger offered rewards; mean wait time on catch trials rises with offer across 5, 10, 20, 40 and 80 uL.

Per rat, mean wait time on opt-out catch trials in mixed blocks per offer (wtcurves.m), then the mean over rats.

In [ ]:
mix_wt = t[(t.block == 1) & t.wt.notna()]
curve = mix_wt.groupby(["subject", "vol"]).wt.mean().unstack()
print("rats with every offer:", int(curve.notna().all(1).sum()), "of", len(curve))
pop = curve.mean()
print("population mean wait time (s) by offer:", pop.round(3).to_dict())
print("increases at every step:", bool((np.diff(pop.to_numpy()) > 0).all()))
print("rats whose own curve increases at every step:", int((np.diff(curve.to_numpy(), axis=1) > 0).all(1).sum()))
print("by sex:", curve.groupby(sex).mean().round(2).to_dict("index"))
R["fig1e"] = pop.round(2).tolist()
fig, ax = plt.subplots(figsize=(4, 3))
for b, nm in [(1, "mixed"), (2, "high"), (3, "low")]:
    c = t[(t.block == b) & t.wt.notna()].groupby(["subject", "vol"]).wt.mean().unstack().mean()
    ax.plot(np.log2(c.index.astype(float)), c.values, "o-", label=nm)
ax.set_xticks(np.log2([5, 10, 20, 40, 80])); ax.set_xticklabels([5, 10, 20, 40, 80])
ax.set_xlabel("offer (uL)"); ax.set_ylabel("mean wait time (s)"); ax.legend(); plt.show()

**Result: reproduced.** The population mean wait time on mixed block catch trials rises at every step from 5 to 80 uL (10.45, 10.77, 11.11, 13.03 and 13.81 s); 171 of 291 rats rise at every step on their own.

## fig1f-wait-time-block-effect

> Rats wait about 10% less for 20 uL in high blocks than in low blocks, across the population.

Ratio per rat of mean wait time on 20 uL opt-out catch trials in high over low blocks. The population test is run on (a) per-rat means of wait times z-scored over all the rat's catch trials (Methods) and (b) raw means (Figure1.m).

In [ ]:
w20 = t[(t.vol == 20) & t.wt.notna() & t.block.isin([2, 3])]
m_raw = w20.groupby(["subject", "block"]).wt.mean().unstack()
m_z = w20.groupby(["subject", "block"]).z_wt.mean().unstack()
ratio = (m_raw[2] / m_raw[3])
print("rats with both blocks:", int(m_raw.notna().all(1).sum()))
print(f"mean ratio high/low = {ratio.mean():.4f} (expected 0.90 +- 0.025); median {ratio.median():.4f}; "
      f"range {ratio.min():.3f}-{ratio.max():.3f}; rats with ratio < 1: {(ratio < 1).sum()}")
p_a = stats.wilcoxon(m_z[2], m_z[3]).pvalue
p_b = stats.wilcoxon(m_raw[2], m_raw[3]).pvalue
print(f"(a) signed-rank on z-scored means: p = {p_a:.3g}; mean z high {m_z[2].mean():.3f}, low {m_z[3].mean():.3f}")
print(f"(b) signed-rank on raw means: p = {p_b:.3g}; mean high {m_raw[2].mean():.3f} s, low {m_raw[3].mean():.3f} s")
pr = w20.groupby("subject").apply(lambda g: M.ranksum_p(g.wt[g.block == 2], g.wt[g.block == 3], "less"))
print("per-rat one-tailed rank-sum (high shorter), rats with p < 0.05:", int((pr < 0.05).sum()), "of", int(pr.notna().sum()))
print("ratio by sex (mean):", ratio.groupby(sex).mean().round(4).to_dict())
R["fig1f"] = (ratio.mean(), p_a, p_b)
wt_ratio = ratio

**Result: reproduced.** The mean ratio of 20 uL wait times, high over low, is 0.894 (expected 0.90 +- 0.025), and the signed-rank test across 291 rats gives p = 4.6e-47 on z-scored means (a) and 8.2e-47 on raw means (b). 244 of 291 rats wait significantly less in high blocks by the per-rat one-tailed rank-sum test.

## fig1i-initiation-block-effect

> Rats initiate trials more quickly in high blocks than in low blocks, across the population.

Per rat, mean trial initiation time over all trials in high and low blocks (iticurves.m); (a) z-scored over all the rat's trials (Methods), (b) raw seconds (Figure1.m).

In [ ]:
hl = t[t.block.isin([2, 3]) & t.iti.notna()]
i_raw = hl.groupby(["subject", "block"]).iti.mean().unstack()
i_z = hl.groupby(["subject", "block"]).z_iti.mean().unstack()
iti_ratio = i_raw[2] / i_raw[3]
pa = stats.wilcoxon(i_z[2], i_z[3]).pvalue
pb = stats.wilcoxon(i_raw[2], i_raw[3]).pvalue
print(f"mean ratio high/low = {iti_ratio.mean():.4f}; median {iti_ratio.median():.4f}; rats < 1: {(iti_ratio < 1).sum()} of {iti_ratio.notna().sum()}")
print(f"(a) z-scored: p = {pa:.3g}; mean z high {i_z[2].mean():.3f}, low {i_z[3].mean():.3f}")
print(f"(b) raw: p = {pb:.3g}; mean high {i_raw[2].mean():.3f} s, low {i_raw[3].mean():.3f} s")
pr = hl.groupby("subject").apply(lambda g: M.ranksum_p(g.iti[g.block == 2], g.iti[g.block == 3]))
print("per-rat two-tailed rank-sum, rats with p < 0.05:", int((pr < 0.05).sum()))
R["fig1i"] = (iti_ratio.mean(), pa, pb)

**Result: reproduced.** The mean initiation time ratio, high over low, is 0.568 and below 1 in all 291 rats; the signed-rank test gives p = 1.8e-49 under both readings (the paper prints 1.82e-49).

## suppfig2-initiation-block-effect-after-unrewarded

> The block sensitivity of trial initiation times comes from trials after unrewarded trials; after rewarded trials initiation times do not differ between high and low blocks in the same direction.

Per rat, mean z-scored initiation time in high and low blocks split by whether the previous trial (same session) was rewarded; unrewarded includes opt-outs and violations. Tests two-tailed (legend) and one-tailed, low longer than high (FigureS2.m).

In [ ]:
res = {}
for name, m in [("post-rewarded", hl.prev_is_rewarded == 1), ("post-unrewarded", hl.prev_is_rewarded == 0),
                ("all", pd.Series(True, index=hl.index))]:
    mm = hl[m].groupby(["subject", "block"]).z_iti.mean().unstack().dropna()
    p2 = stats.wilcoxon(mm[2], mm[3]).pvalue
    p1 = stats.wilcoxon(mm[3], mm[2], alternative="greater").pvalue
    d = (mm[2] - mm[3])
    res[name] = (p2, p1, d.mean())
    print(f"{name:16s} n={len(mm)} mean z high {mm[2].mean():+.3f} low {mm[3].mean():+.3f}; high-low {d.mean():+.3f}; "
          f"rats high<low {(d < 0).sum()}; two-tailed p = {p2:.3g}; one-tailed (low > high) p = {p1:.3g}")
R["suppfig2"] = res

**Result: reproduced.** After unrewarded trials initiation is faster in high blocks in all 291 rats (two-tailed p = 1.8e-49; one-tailed p = 9.0e-50, the paper prints 9.07e-50). After rewarded trials the one-tailed test in that direction gives p = 1, as printed in the paper; the two-tailed test finds a small difference the other way (high minus low +0.018 z, p = 8.5e-10), which the criterion does not count against the claim.

## suppfig3-wait-time-not-affected-by-previous-outcome

> Wait times show no major differences after rewarded and after unrewarded trials.

Wait time ratio (20 uL, high over low) per rat on catch trials after rewarded and after unrewarded trials.

In [ ]:
out = {}
for name, v in [("after rewarded", 1), ("after unrewarded", 0)]:
    mm = w20[w20.prev_is_rewarded == v].groupby(["subject", "block"]).wt.mean().unstack()
    out[name] = mm[2] / mm[3]
    r = out[name]
    print(f"{name:17s} rats {r.notna().sum()}: mean ratio {r.mean():.4f}, median {r.median():.4f}")
d = (out["after rewarded"] - out["after unrewarded"]).dropna()
print(f"difference rewarded - unrewarded: mean {d.mean():+.4f}, n {len(d)}, signed-rank p = {stats.wilcoxon(d).pvalue:.3g} (variant)")
R["suppfig3"] = (out["after rewarded"].mean(), out["after unrewarded"].mean(), d.mean())

**Result: reproduced.** The mean wait time ratio is 0.899 after rewarded and 0.897 after unrewarded trials, both below 1; they differ by 0.002 (signed-rank p = 0.63, a variant).

## suppfig12-no-sex-differences

> Male and female rats do not differ in their wait time ratios or trial initiation time ratios.

Per-rat ratios of fig1f and fig1i (raw), rank-sum test between sexes (FigureS12.m). A claim of absence.

In [ ]:
s12 = {}
for name, r in [("wait time ratio", wt_ratio), ("initiation time ratio", iti_ratio)]:
    a, b = r[sex.reindex(r.index) == "M"].dropna(), r[sex.reindex(r.index) == "F"].dropna()
    p = M.ranksum_p(a, b)
    s12[name] = p
    print(f"{name}: males n={len(a)} mean {a.mean():.4f}, females n={len(b)} mean {b.mean():.4f}; rank-sum p = {p:.3g}")
R["suppfig12"] = s12

**Result: reproduced.** Neither ratio differs between the 184 males and 107 females: rank-sum p = 0.238 for the wait time ratio (the paper prints 0.23) and p = 0.343 for the initiation time ratio (the paper prints 0.59).

## suppfig14-contextual-effects-without-threshold

> Without removing wait times more than 1 SD above the pooled mean, the contextual effects on wait times are qualitatively similar, with longer average wait times, mainly in low blocks.

In [ ]:
ok = t.wt_raw.notna()
cur_raw = t[ok].groupby(["subject", "block", "vol"]).wt_raw.mean().unstack().groupby("block").mean()
cur_thr = t[t.wt.notna()].groupby(["subject", "block", "vol"]).wt.mean().unstack().groupby("block").mean()
print("unthresholded (s):\n", cur_raw.round(2), "\nthresholded (s):\n", cur_thr.round(2))
diff = (cur_raw - cur_thr)
print("unthresholded minus thresholded (s):\n", diff.round(2))
cells = diff.stack().dropna()
print("cells with longer unthresholded mean:", int((cells > 0).sum()), "of", len(cells))
u20 = t[ok & (t.vol == 20) & t.block.isin([2, 3])]
mz = u20.groupby(["subject", "block"]).wt_raw.mean().unstack()
zz = zscore_raw = M.zscore_within(t, "wt_raw", ok)
mzz = zz[u20.index].groupby([u20.subject, u20.block]).mean().unstack()
print(f"20 uL high vs low, unthresholded: raw means high {mz[2].mean():.3f} low {mz[3].mean():.3f}, signed-rank p = {stats.wilcoxon(mz[2], mz[3]).pvalue:.3g}; "
      f"z-scored p = {stats.wilcoxon(mzz[2], mzz[3]).pvalue:.3g}; mean ratio {(mz[2]/mz[3]).mean():.4f}")
print("mean increase by block (s):", diff.mean(1).round(2).to_dict(), "(1 mixed, 2 high, 3 low)")
R["suppfig14"] = (int((cells > 0).sum()), len(cells), stats.wilcoxon(mzz[2], mzz[3]).pvalue)

**Result: reproduced.** Without the threshold, 20 uL wait times are still shorter in high than in low blocks (signed-rank p = 1.5e-44 on z-scored means, 1.0e-43 on raw means), and the population mean is longer unthresholded in all 11 block and offer cells, most in low blocks (+2.61 s, against +1.42 s in mixed and +1.34 s in high blocks).

## fig2a-wait-time-converges-in-mixed-blocks

> After a transition into a mixed block, wait times converge to a common value whether the previous block was high or low, while trial initiation times in mixed blocks keep depending on the previous block (slower after high blocks, faster after low blocks).

Curves per rat over trials -20 to 20 around each transition (0 = first trial of the new block; offsets count all trials, violations included), within the session; each rat's curve smoothed with a 10-point moving average, then averaged over rats (Methods, Block transition dynamics). Wait times: opt-out catch trials that are not violations, z-scored within rat, then within each offer (deltawt in block_dynamics_wt.m). Initiation times: z-scored within rat and detrended against trial number, per session (Methods) or with one line over pooled sessions (block_dynamics_latency.m). The code also overwrites the value at the transition trial with a neighbour; that plotting step is left out here.

In [ ]:
cw = t.wt.notna() & ~t.is_violation
zw = M.zscore_within(t, "wt", cw)
t["zv_wt"] = M.zscore_within(t.assign(zw=zw), "zw", cw & zw.notna(), keys=("subject", "vol"))
t["iti_dt_sess"] = M.detrend_iti(t, t.z_iti, per_session=True)
t["iti_dt_pool"] = M.detrend_iti(t, t.z_iti, per_session=False)
trans = M.transitions(t)
print("transitions by type (prev -> next):", trans.groupby(["prev", "next"]).size().to_dict())
curves = {k: M.transition_curves(t, t[k], trans, rats=rats) for k in ["zv_wt", "iti_dt_sess", "iti_dt_pool", "z_iti"]}
offs = np.arange(-20, 21)
HM, LM, MH, ML = (2, 1), (3, 1), (1, 2), (1, 3)

def gap(c, a, b, lo, hi):
    """mean over offsets lo..hi of |population curve a - population curve b|, and the signed mean"""
    d = (c[a].mean() - c[b].mean()).loc[lo:hi]
    return float(d.abs().mean()), float(d.mean())

fig2a = {}
for k in ["iti_dt_sess", "iti_dt_pool"]:
    for lo in [1, 10]:
        gw_post, _ = gap(curves["zv_wt"], HM, LM, lo, 20)
        gw_pre, sw_pre = gap(curves["zv_wt"], HM, LM, -20, -1)
        gi_post, si_post = gap(curves[k], HM, LM, lo, 20)
        gi_pre, si_pre = gap(curves[k], HM, LM, -20, -1)
        rw, ri = gw_post / gw_pre, gi_post / gi_pre
        fig2a[(k, lo)] = (rw, ri)
        print(f"[{k}, trials {lo}-20] wait time gap post/pre = {gw_post:.3f}/{gw_pre:.3f} = {rw:.3f}; "
              f"initiation gap post/pre = {gi_post:.3f}/{gi_pre:.3f} = {ri:.3f} (signed post {si_post:+.3f}); part 1 holds: {rw < ri}")

# part 2: mean detrended z-scored initiation time over the whole mixed block, after high vs after low
b = t.block.to_numpy(); s = t.sess.to_numpy()
start = np.r_[True, (b[1:] != b[:-1]) | (s[1:] != s[:-1])]
blk_id = np.cumsum(start)
first = np.flatnonzero(start)
prevb = np.r_[0, np.where(s[first[1:]] == s[first[1:] - 1], b[first[1:] - 1], 0)]
t["prev_block_of_block"] = prevb[blk_id - 1]
mixed = t[t.block == 1]
part2 = {}
for k in ["iti_dt_sess", "iti_dt_pool", "z_iti"]:
    mm = mixed.groupby(["subject", "prev_block_of_block"])[k].mean().unstack()[[2, 3]].dropna()
    p = stats.wilcoxon(mm[2], mm[3]).pvalue
    part2[k] = (mm[2].mean(), mm[3].mean(), p, len(mm))
    print(f"part 2 [{k}]: mixed block after high {mm[2].mean():+.3f}, after low {mm[3].mean():+.3f}; "
          f"rats after-high > after-low {(mm[2] > mm[3]).sum()} of {len(mm)}; signed-rank p = {p:.3g}")
print("by sex (iti_dt_sess, after high minus after low):",
      (mixed.groupby(["subject", "prev_block_of_block"]).iti_dt_sess.mean().unstack().pipe(lambda m: m[2] - m[3]).groupby(sex).mean().round(3).to_dict()))
R["fig2a"] = (fig2a, part2)

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, k, nm in [(axs[0], "zv_wt", "wait time (z, per offer)"), (axs[1], "iti_dt_sess", "initiation time (z, detrended)")]:
    for tr, lab in [(HM, "high->mixed"), (LM, "low->mixed")]:
        ax.plot(offs, curves[k][tr].mean().values, label=lab)
    ax.axvline(0, color="k", lw=0.5); ax.set_title(nm); ax.set_xlabel("trial from transition")
axs[0].legend(); plt.show()
for k in ["zv_wt", "iti_dt_sess"]:
    for tr in [HM, LM, MH, ML]:
        v = curves[k][tr].mean()
        print(f"plotted range {k} {tr}: {v.min():.3f} to {v.max():.3f}")

**Result: reproduced.** After a transition into a mixed block, the gap between the wait time curves after high and after low blocks is 0.341 of its pre-transition size over trials 1-20 (0.086 over trials 10-20), while the initiation time gap is 1.90 times its pre-transition size (2.10 over trials 10-20; 2.22 and 2.45 with the code's pooled detrending). Over the whole mixed block, initiation is slower after high than after low blocks (+0.048 against -0.102 z, 290 of 291 rats, signed-rank p = 1.8e-49; with pooled detrending 291 rats, p = 1.8e-49).

## suppfig5-mixed-to-adaptation-transitions

> Longer timescale contrast effects also appear at transitions from mixed into high or low blocks for trial initiation times, but not for wait times.

The same curves for mixed-to-high and mixed-to-low transitions.

In [ ]:
s5 = {}
for k in ["iti_dt_sess", "iti_dt_pool"]:
    gi_pre, si_pre = gap(curves[k], ML, MH, -20, -1)
    gi_post, _ = gap(curves[k], ML, MH, 1, 20)
    gw_pre, sw_pre = gap(curves["zv_wt"], ML, MH, -20, -1)
    gw_post, _ = gap(curves["zv_wt"], ML, MH, 1, 20)
    s5[k] = (si_pre, gw_pre / gw_post, gi_pre / gi_post)
    print(f"[{k}] trials -20..-1: initiation (before low) - (before high) = {si_pre:+.3f} (must be > 0); "
          f"gap pre/post: wait time {gw_pre:.3f}/{gw_post:.3f} = {gw_pre/gw_post:.3f}, initiation {gi_pre:.3f}/{gi_post:.3f} = {gi_pre/gi_post:.3f}; "
          f"holds: {si_pre > 0 and gw_pre/gw_post < gi_pre/gi_post}")
print(f"wait time signed difference before low minus before high over -20..-1: {sw_pre:+.3f}")
R["suppfig5"] = s5
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, k in [(axs[0], "zv_wt"), (axs[1], "iti_dt_sess")]:
    for tr, lab in [(MH, "mixed->high"), (ML, "mixed->low")]:
        ax.plot(offs, curves[k][tr].mean().values, label=lab)
    ax.axvline(0, color="k", lw=0.5); ax.set_title(k)
axs[0].legend(); plt.show()

**Result: reproduced.** In the mixed block before a transition, initiation times are higher before low blocks than before high blocks by +0.046 z (+0.059 with pooled detrending), and the wait time curves differ there by 0.125 of their post-transition gap, against 0.197 (0.239) for initiation times.

## suppfig6-results-without-detrending

> Satiety effects on trial initiation times are modest, and the results, including the dynamics at block transitions, do not change qualitatively without detrending.

The initiation time curves without the trial number regression (z-scored only).

In [ ]:
gi_post, si_post = gap(curves["z_iti"], HM, LM, 1, 20)
p2 = part2["z_iti"]
print(f"no detrending: high->mixed minus low->mixed over trials 1-20 = {si_post:+.3f} (ordering holds if > 0)")
print(f"part 2 without detrending: after high {p2[0]:+.3f}, after low {p2[1]:+.3f}, p = {p2[2]:.3g}")
pre_h = curves["z_iti"][HM].mean().loc[-20:-1].mean(); pre_l = curves["z_iti"][LM].mean().loc[-20:-1].mean()
print(f"trials -20..-1: in high block {pre_h:+.3f}, in low block {pre_l:+.3f}")
slope = t.groupby("sess").apply(lambda g: np.polyfit(g.trial[g.z_iti.notna()], g.z_iti.dropna(), 1)[0] if g.z_iti.notna().sum() > 2 else np.nan)
print(f"satiety: per-session slope of z-scored initiation time on trial number, median {slope.median()*100:.4f} z per 100 trials")
R["suppfig6"] = (si_post, p2)

**Result: reproduced.** Without detrending the high-to-mixed initiation curve lies above the low-to-mixed curve over trials 1-20 by +0.246 z, and the mixed-block test gives +0.094 against -0.085 z, p = 1.8e-49. The median per-session trend of z-scored initiation time is 0.069 z per 100 trials.

## fig2c-d-regression-on-previous-rewards

> In mixed blocks, wait times are explained by the current offer and barely by previous offers, while trial initiation times reflect an exponentially decaying effect of previous rewards (larger recent rewards, faster initiation).

Per rat, mixed blocks only, log2 offers, offers of a trial in another block (or another session) NaN, OLS with a constant, rows with NaN dropped (Matlab regress). Wait time: z-scored opt-out catch trial wait times in mixed blocks on the current offer and 9 previous offers (violation trials count as previous trials). Initiation time: initiation times of mixed block trials, z-scored over those trials, on previous log2 rewards times hit (unrewarded trials 0; regress_latency_vs_rew.m), 9 back (Methods) or 10 back (code). Lags are taken within the session; the code lags over concatenated sessions, which differs only at session starts (a standard choice).

In [ ]:
def rat_regressions(g, nback_iti):
    sess = g.sess.to_numpy(); blk = g.block.to_numpy()
    logv = np.log2(g.vol.to_numpy().astype(float))
    mixed = blk == 1
    lv = np.where(mixed, logv, np.nan)
    # wait time
    cw = mixed & g.wt.notna().to_numpy()
    y = g.wt.to_numpy()
    yz = np.full(len(g), np.nan)
    yz[cw] = (y[cw] - y[cw].mean()) / y[cw].std(ddof=1)
    Xw = np.c_[lv, M.lagged(lv, sess, 9), np.ones(len(g))]
    bw, ciw, nw = M.ols_ci(Xw[cw], yz[cw])
    # initiation time
    rr = np.where(mixed, logv * g.is_rewarded.to_numpy(), np.nan)
    it = g.iti.to_numpy()
    ci = mixed & ~np.isnan(it)
    iz = np.full(len(g), np.nan)
    iz[ci] = (it[ci] - it[ci].mean()) / it[ci].std(ddof=1)
    Xi = np.c_[M.lagged(rr, sess, nback_iti), np.ones(len(g))]
    bi, cii, ni = M.ols_ci(Xi[mixed], iz[mixed])
    return bw, ciw, bi, cii

REG = {}
for nb in [9, 10]:
    out = {r: rat_regressions(g, nb) for r, g in t.groupby("subject", sort=True)}
    REG[nb] = out
bw = np.array([REG[9][r][0] for r in rats]); ciw = np.array([REG[9][r][1] for r in rats])
print("wait time coefficients (current, 1..9 back, constant), mean over rats:\n", np.round(np.nanmean(bw, 0), 4))
cur = bw[:, 0]
m, se = np.nanmean(cur), np.nanstd(cur, ddof=1) / np.sqrt(np.isfinite(cur).sum())
print(f"current offer: mean {m:.4f}, 95% interval across rats {m-1.96*se:.4f} to {m+1.96*se:.4f}; rats with own CI excluding 0 and > 0: {int(((ciw[:,0,0] > 0)).sum())}")
prev_abs = np.abs(np.nanmean(bw[:, 1:10], 0))
print("largest |mean previous coefficient|:", prev_abs.max().round(4), "; current larger than every previous:", bool(abs(m) > prev_abs.max()))
fig2cd = {}
for nb in [9, 10]:
    bi = np.array([REG[nb][r][2] for r in rats])
    mi = np.nanmean(bi[:, :nb], 0)
    ok = mi[0] < 0 and mi[1] < 0 and mi[2] < 0 and abs(mi[0]) > abs(mi[1]) > abs(mi[2])
    fig2cd[nb] = (mi, ok)
    print(f"initiation coefficients ({nb} back), mean over rats:", np.round(mi, 4), "| 1-3 back negative and decreasing in size:", ok)
    print("   rats with 1-back coefficient < 0:", int((bi[:, 0] < 0).sum()), "of", len(bi))
R["fig2cd"] = (m, m - 1.96 * se, m + 1.96 * se, prev_abs.max(), fig2cd)
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].errorbar(np.arange(0, 10), np.nanmean(bw[:, :10], 0), np.nanstd(bw[:, :10], 0) / np.sqrt(len(bw)), fmt="o")
axs[0].set_title("wait time"); axs[0].set_xlabel("trials back (0 = current)")
axs[1].errorbar(np.arange(1, 10), fig2cd[9][0], fmt="o"); axs[1].set_title("initiation time"); plt.show()

**Result: reproduced.** The current offer coefficient for wait time is 0.212 (95% interval across rats 0.205 to 0.219), larger than every previous offer coefficient (largest 0.013 in size). The initiation time coefficients 1, 2 and 3 trials back are -0.029, -0.024 and -0.012 with 9 lags, and the same to three decimals with 10.

## fig2e-tau-differ

> The time constants of exponential fits to the previous trial coefficients differ between trial initiation times and wait times, with longer time constants for initiation times (wait time time constants are mostly 0).

Per rat, the first coefficient whose 95% interval contains 0 and all after it are set to 0; y = D exp(-x/tau) fitted to the previous trial coefficients. Methods reading: the 9 previous coefficients of both regressions (initiation with 9 back), wait time coefficients times 10, D in [-10, 10], tau in [0, 10]; initiation D in [-10, 0], tau in [0, 10]; tau = 0 when all previous coefficients are 0. Code reading: wait time fits coefficients 1 to 10 (current and 9 previous) with the zeroing started at the current offer; initiation uses 10 back and tau is 0 when coefficients 2 to 10 are all 0. The fit is a dense grid over tau (see the helper), so no random starts are drawn.

In [ ]:
def zero_after_first_ns(b, ci, start, stop):
    b = b.copy()
    ns = np.flatnonzero((ci[start:stop, 0] < 0) & (ci[start:stop, 1] > 0))
    if len(ns):
        b[start + ns[0]:stop] = 0
    return b

TAU = {}
for reading in ["methods", "code"]:
    nb = 9 if reading == "methods" else 10
    W, I = [], []
    for r in rats:
        bw_, ciw_, bi_, cii_ = REG[nb][r]
        if reading == "methods":
            W.append(zero_after_first_ns(bw_, ciw_, 1, 10)[1:10])
        else:
            W.append(zero_after_first_ns(bw_, ciw_, 0, 10)[0:10])
        I.append(zero_after_first_ns(bi_, cii_, 0, nb)[:nb])
    W, I = np.nan_to_num(np.array(W)), np.nan_to_num(np.array(I))
    xw = np.arange(1, W.shape[1] + 1); xi = np.arange(1, nb + 1)
    _, tw = M.fit_exp_decay(xw, 10 * W, (-10, 0), (10, 10))
    _, ti = M.fit_exp_decay(xi, I, (-10, 0), (0, 10))
    tw = np.where(W.any(1), tw, 0.0)
    ti = np.where(I.any(1) if reading == "methods" else I[:, 1:].any(1), ti, 0.0)
    p = stats.wilcoxon(ti, tw).pvalue
    TAU[reading] = (tw, ti, W, I)
    print(f"[{reading}] wait time tau: mean {tw.mean():.3f}, median {np.median(tw):.3f}, zero in {int((tw == 0).sum())} of {len(tw)}; "
          f"initiation tau: mean {ti.mean():.3f}, median {np.median(ti):.3f}, zero in {int((ti == 0).sum())}; signed-rank p = {p:.3g}; "
          f"at the upper bound 10: wait {int((tw >= 9.999).sum())}, initiation {int((ti >= 9.999).sum())}")
    R.setdefault("fig2e", {})[reading] = (tw.mean(), ti.mean(), p, int((tw == 0).sum()))

**Result: reproduced.** Under the Methods reading the mean tau is 0.065 for wait times (0 in 249 of 291 rats) and 1.875 for initiation times, signed-rank p = 3.5e-45; under the code reading 0.620 and 1.914, p = 2.7e-35. Six initiation taus (seven under the code reading) sit at the upper bound of 10.

## fig2f-tau-uncorrelated

> Across rats, the time constants for wait times and trial initiation times are not correlated.

Pearson correlation of the per-rat taus of fig2e (rows complete). A claim of absence: reproduced when p >= 0.05.

In [ ]:
for reading in ["methods", "code"]:
    tw, ti = TAU[reading][:2]
    r_, p_ = stats.pearsonr(tw, ti)
    nz = (tw > 0)
    print(f"[{reading}] r = {r_:.3f}, p = {p_:.3g} (paper: r = 0.08, p = 0.18 in the Results; r = -0.03, p = 0.53 in the legend); "
          f"rats with nonzero wait time tau: {int(nz.sum())}")
    R.setdefault("fig2f", {})[reading] = (r_, p_)

**Result: reproduced.** The taus are not correlated: r = -0.037, p = 0.535 under the Methods reading (the legend prints r = -0.03, p = 0.53) and r = 0.111, p = 0.059 under the code reading.

## fig2g-h-fast-slow-tau-groups

> Rats split by fast or slow temporal integration of initiation times (tau below the 20th or above the 80th percentile) differ in their initiation time regression coefficients but not in their wait time coefficients.

myTauShuffleTest.m: the rats below the 20th and above the 80th percentile of initiation tau; observed fits to each group's mean coefficients (D in [-10, 1], tau in [0, 10]); 500 shuffles of the labels, each half's mean coefficients times 10 refitted (D in [-3, 1], tau in [0, 50]); p = share of shuffles with |tau difference| above the observed, divided by 2. The code passes all coefficients but the constant (initiation 1 to 10 back; wait time current and 9 back); the code reading of fig2e is used for the coefficients, and the Methods reading is shown too. Seeds: the claim's and +1000 to +4000.

In [ ]:
def tau_shuffle(B, first_q, last_q, rng, nshuf=500):
    x = np.arange(1, B.shape[1] + 1)
    sel = first_q | last_q
    Bs = B[sel]; n = len(Bs)
    perms = np.array([rng.permutation(n) for _ in range(nshuf)])
    h = n // 2
    g1 = Bs[perms[:, :h]].mean(1); g2 = Bs[perms[:, h:]].mean(1)
    _, t1 = M.fit_exp_decay(x, 10 * g1, (-3, 0), (1, 50))
    _, t2 = M.fit_exp_decay(x, 10 * g2, (-3, 0), (1, 50))
    _, o1 = M.fit_exp_decay(x, B[first_q].mean(0)[None], (-10, 0), (1, 10))
    _, o2 = M.fit_exp_decay(x, B[last_q].mean(0)[None], (-10, 0), (1, 10))
    obs = o1[0] - o2[0]
    return np.sum(np.abs(t1 - t2) > abs(obs)) / (2 * nshuf), obs

for reading in ["code", "methods"]:
    tw, ti, W, I = TAU[reading]
    lo, hi = np.percentile(ti, 20), np.percentile(ti, 80)
    fq, lq = ti < lo, ti > hi
    print(f"[{reading}] initiation tau 20th pct {lo:.3f}, 80th {hi:.3f}; fast group {fq.sum()}, slow group {lq.sum()} (paper N = 116)")
    for off in ([0, 1000, 2000, 3000, 4000] if reading == "code" else [0]):
        rng = rng_for("fig2g-h-fast-slow-tau-groups", offset=off)
        pi, oi = tau_shuffle(I, fq, lq, rng)
        pw, ow = tau_shuffle(W, fq, lq, rng)
        print(f"   seed +{off}: initiation p = {pi:.3f} (observed tau diff {oi:+.3f}); wait time p = {pw:.3f} (observed {ow:+.3f})")
        R.setdefault("fig2gh", {})[(reading, off)] = (pi, pw)

**Result: reproduced.** The 20th and 80th percentiles give 58 rats per group (116 in all, as in the paper). With the code's coefficients the initiation time p is 0.000 under all five seeds and the wait time p is 0.203 to 0.217; with the Methods' coefficients 0.000 and 0.394.

## fig2j-wait-time-not-sensitive-to-previous-offer

> In mixed blocks, wait times for 20 uL do not depend on whether the previous offer was smaller or larger than 20 uL, for 87% of rats (253 of 291) and for the population.

Per rat, 20 uL opt-out catch trials in mixed blocks that are not the first trial of a session, wait times z-scored over these trials, split by the previous trial's offer below or above 20 uL (waittime_20ul_by_previous_vol.m), two-tailed rank-sum. Population: per-rat means, two-tailed rank-sum (legend); paired signed-rank (Figure2.m) as a variant.

In [ ]:
sel = (t.vol == 20) & (t.block == 1) & t.wt.notna() & t.prev_vol.notna()
w = t[sel]
w = w.assign(zw20=w.groupby("subject").wt.transform(lambda x: (x - x.mean()) / x.std()))
lt, gt = w.prev_vol < 20, w.prev_vol > 20
pj = w.groupby("subject").apply(lambda g: M.ranksum_p(g.zw20[g.prev_vol < 20], g.zw20[g.prev_vol > 20]))
wj = pd.DataFrame({"below": w[lt].groupby("subject").zw20.mean(), "above": w[gt].groupby("subject").zw20.mean()})
n_sig = int((pj < 0.05).sum())
p_pop = M.ranksum_p(wj.below, wj.above); p_sr = stats.wilcoxon(wj.below, wj.above).pvalue
sens_wt = wj.below - wj.above
print(f"rats with p < 0.05: {n_sig} of {pj.notna().sum()} (expected 38 +- 4); not significant: {pj.notna().sum() - n_sig}")
print(f"population: mean z after < 20 uL {wj.below.mean():+.4f}, after > 20 uL {wj.above.mean():+.4f}; rank-sum p = {p_pop:.3g} (paper 0.8); signed-rank p = {p_sr:.3g} (variant)")
print(f"sensitivity (below minus above): mean {sens_wt.mean():+.4f}; trials per rat median {w.groupby('subject').size().median():.0f}")
print("rats with p < 0.05 by sex:", (pj < 0.05).groupby(sex).sum().to_dict())
R["fig2j"] = (n_sig, p_pop, p_sr, sens_wt.mean())

**Result: reproduced.** 38 of 291 rats have p < 0.05 (expected 38 +- 4; the other 253 are the paper's count), and the population rank-sum on per-rat means gives p = 0.575 (signed-rank variant p = 0.629).

## fig2k-initiation-sensitive-to-previous-offer

> In mixed blocks, trial initiation times are slower after offers smaller than 20 uL than after offers larger than 20 uL, significantly so for 89% of rats (256 of 291) and for the population.

Readings: (M) Methods, 20 uL trials in mixed blocks; (C) latency_post20ul_by_previous_vol.m, all mixed block trials; both with initiation times z-scored over all the rat's trials. Previous offer within the session. Per rat two-tailed rank-sum; population rank-sum on per-rat means (legend) and signed-rank (Figure2.m).

The two readings give opposite directions, so the cell also splits the code's trials by whether the previous trial was a violation. After a violation the task repeats the offer (Methods, Pre-initiation cue task: "those trials repeat the same volume as the previous trial so the rat could conceivably use that to modulate their trial initiation time"), so on those trials the previous offer is also the coming one. A 20 uL trial with a previous offer other than 20 uL can never follow a violation, so reading (M) holds no post-violation trials.

In [ ]:
FIG2K = {}
# post-violation trials: the previous trial in the session was a violation. The task repeats the
# offer after a violation (Methods, Pre-initiation cue task: "those trials repeat the same volume
# as the previous trial"), so on these trials the rat can know the coming offer.
t["post_violation"] = (t.is_violation.shift(1).where(t.trial > 0) == True).to_numpy()
pv_ = t.post_violation & t.prev_vol.notna()
print(f"violation trials: {t.is_violation.mean():.3f} of all trials; offer equal to the previous offer: after a violation "
      f"{(t.vol[pv_] == t.prev_vol[pv_]).mean():.4f} of {int(pv_.sum())} trials, after any other trial "
      f"{(t.vol[~t.post_violation & t.prev_vol.notna()] == t.prev_vol[~t.post_violation & t.prev_vol.notna()]).mean():.4f}")

def prev_offer_test(m):
    sel = m & (t.block == 1) & t.z_iti.notna() & t.prev_vol.notna()
    d = t.loc[sel, ["subject", "z_iti", "prev_vol"]]
    pk = d.groupby("subject").apply(lambda g: M.ranksum_p(g.z_iti[g.prev_vol < 20], g.z_iti[g.prev_vol > 20]))
    mk = pd.DataFrame({"below": d[d.prev_vol < 20].groupby("subject").z_iti.mean(), "above": d[d.prev_vol > 20].groupby("subject").z_iti.mean()})
    return d, pk, mk

ALLT = pd.Series(True, index=t.index)
for name, m in [("M: 20 uL trials", (t.vol == 20)), ("C: all mixed trials", ALLT)]:
    d, pk, mk = prev_offer_test(m)
    nsig = int((pk < 0.05).sum()); nsig_dir = int(((pk < 0.05) & (mk.below > mk.above)).sum())
    pp, ps = M.ranksum_p(mk.below, mk.above), stats.wilcoxon(mk.below, mk.above).pvalue
    FIG2K[name] = mk.below - mk.above
    R.setdefault("fig2k", {})[name] = (nsig, pp, ps, mk.below.mean(), mk.above.mean())
    print(f"[{name}] rats p < 0.05: {nsig} (expected 256 +- 26), of which slower after small offers {nsig_dir}; "
          f"mean z after < 20 {mk.below.mean():+.4f}, after > 20 {mk.above.mean():+.4f}; rank-sum p = {pp:.3g}; signed-rank p = {ps:.3g}")
    if name.startswith("C"):
        print(f"   share of post-violation trials: previous offer < 20 uL {t.post_violation[d.index][d.prev_vol < 20].mean():.3f}, "
              f"> 20 uL {t.post_violation[d.index][d.prev_vol > 20].mean():.3f}")
    else:
        print(f"   post-violation trials with a previous offer other than 20 uL: {int(t.post_violation[d.index][d.prev_vol != 20].sum())} "
              "(the offer repeats, so none enter the comparison)")
# where the code's result comes from: its trials split by whether the previous trial was a violation,
# and the trials after other trials split by the previous outcome
for name, m in [("C, post-violation trials only", t.post_violation),
                ("C, post-violation trials left out", ~t.post_violation),
                ("C, left out, previous trial rewarded", ~t.post_violation & (t.prev_is_rewarded == True)),
                ("C, left out, previous trial unrewarded (opt-out)", ~t.post_violation & (t.prev_is_rewarded == False))]:
    d, pk, mk = prev_offer_test(m)
    pp = M.ranksum_p(mk.below, mk.above)
    FIG2K[name] = mk.below - mk.above
    print(f"[{name}] trials {len(d)}; rats p < 0.05: {int((pk < 0.05).sum())}, slower after small offers "
          f"{int(((pk < 0.05) & (mk.below > mk.above)).sum())}; mean z after < 20 {mk.below.mean():+.4f}, after > 20 {mk.above.mean():+.4f}; rank-sum p = {pp:.3g}")
    R.setdefault("fig2k_pv", {})[name] = (int((pk < 0.05).sum()), int(((pk < 0.05) & (mk.below > mk.above)).sum()), mk.below.mean(), mk.above.mean(), pp)

**Result: partially reproduced.** With all mixed block trials (the code's reading), 256 rats have p < 0.05 (expected 256 +- 26), 255 of them slower after small offers, and the population is slower after small offers (+0.091 against -0.148 z; rank-sum p = 9.9e-97, signed-rank p = 1.9e-49). With the Methods' 20 uL trials, 276 rats have p < 0.05 but only 17 in the claimed direction, and the population is faster after small offers (-0.161 against -0.094 z). The code's result comes from the trials after violations, where the offer repeats in 100% of trials: on those trials all 291 rats are slower after small offers (+0.370 against -0.249 z), while the code's trials without them give 287 significant rats of which 20 are slower after small offers, and a population faster after small offers (-0.164 against -0.096 z, p = 2.4e-32), as in the Methods' reading. Post-violation trials are 51% of the code's trials after offers below 20 uL and 30% after offers above. So the count and direction reproduce only with the trials on which the rat can know the coming offer.

## fig2l-sensitivity-greater-for-initiation

> Sensitivity to the previous offer is larger for trial initiation times than for wait times across rats.

Sensitivity = mean for previous offer below 20 uL minus mean above, per rat, from fig2j and fig2k (each reading); paired signed-rank.

In [ ]:
for name, si in FIG2K.items():
    both = pd.DataFrame({"wt": sens_wt, "iti": si}).dropna()
    p = stats.wilcoxon(both.iti, both.wt).pvalue
    print(f"[{name}] mean sensitivity initiation {both.iti.mean():+.4f}, wait time {both.wt.mean():+.4f}; rats initiation > wait {int((both.iti > both.wt).sum())} of {len(both)}; signed-rank p = {p:.3g}")
    R.setdefault("fig2l", {})[name] = (both.iti.mean(), both.wt.mean(), p)

**Result: partially reproduced.** Under the code's reading of fig2k, initiation time sensitivity (+0.239) exceeds wait time sensitivity (-0.009) in 245 of 291 rats, signed-rank p = 1.6e-36. Under the Methods' reading the initiation time sensitivity is negative (-0.067) and below the wait time sensitivity (103 of 291 rats above it), though the test gives p = 2.2e-06; the code's trials without post-violation trials give the same (-0.067, 100 rats above, p = 2.1e-06). Only the post-violation trials give a larger initiation time sensitivity (+0.619, 287 rats, p = 6.6e-49).

## fig3c-wait-time-by-catch-probability

> Rats' wait times in mixed blocks depend in a graded way on the catch probability, as the wait time model predicts (shorter waits at higher catch probability).

Each rat's first 1000 deposited trials are left out. Figure3.m (lines 66 to 77 at b7b68ca8c107) loads `A_Structs_Final` and sets `dontdrop(1:1000) = false`; the deposit's README describes `A_Structs_Final` as the sessions "passing wait time sensitivity criteria", the sessions the dandiset holds (data check), so the trials dropped here are the ones the code drops. Per rat and catch probability (the trial's `catch_percentage`), mean wait time on mixed block opt-out catch trials by offer, then the mean over rats with data at that probability. Rats are counted at every probability they have data at (reading A), and, as a second reading (B), each rat is assigned to the probability of most of its remaining trials.

In [ ]:
pos = t.groupby("subject").cumcount()
late = t[(pos >= 1000)]
mw = late[(late.block == 1) & late.wt.notna()]
cur = mw.groupby(["catch_percentage", "subject", "vol"]).wt.mean().unstack()
counts_a = cur.groupby(level=0).size()
main_cp = late.groupby("subject").catch_percentage.agg(lambda x: x.value_counts().idxmax())
counts_b = main_cp.value_counts().sort_index()
print("rats with data at each probability (A):", counts_a.to_dict())
print("rats by main probability (B):", counts_b.to_dict())
print("expected: {0.1: 3, 0.15: 183, 0.2: 61, 0.25: 151, 0.35: 39}")
popA = cur.groupby(level=0).mean()
print("A: population mean wait time (s) by probability and offer:\n", popA.round(2))
curB = mw.assign(cp=mw.subject.map(main_cp))
curB = curB[curB.catch_percentage == curB.cp].groupby(["cp", "subject", "vol"]).wt.mean().unstack().groupby(level=0).mean()
print("B: by main probability:\n", curB.round(2))
for nm, P in [("A", popA), ("B", curB)]:
    if all(c in P.index for c in [0.15, 0.2, 0.25]):
        dec = ((P.loc[0.15] > P.loc[0.2]) & (P.loc[0.2] > P.loc[0.25]))
        print(f"{nm}: offers where wait time decreases 0.15 > 0.2 > 0.25: {int(dec.sum())} of 5", dec.to_dict())
        R.setdefault("fig3c", {})[nm] = int(dec.sum())
R["fig3c_counts"] = (counts_a.to_dict(), counts_b.to_dict())
# rats with several probabilities
print("rats with data at more than one probability:", int((mw.groupby('subject').catch_percentage.nunique() > 1).sum()))

**Result: reproduced.** Counting each rat at every probability it has data at (reading A), the cohorts are 3, 188, 61, 151 and 40 rats at 0.1, 0.15, 0.2, 0.25 and 0.35, all within tolerance, and wait times fall from 0.15 to 0.2 to 0.25 at all 5 offers. Assigning each rat to its main probability (reading B) gives 105, 43 and 128 rats at 0.15 to 0.25 and a fall at 3 of 5 offers. Eight rats have data at a probability of 0.3, which the paper does not list.

## bayes-observer-offer-reduces-uncertainty

> In simulations of a Bayes optimal observer, hearing the reward offer reduces the uncertainty of beliefs about the hidden block (posterior variance below prior variance).

Prior (Equation 5) and posterior (Equation 3) of the inferential model with H0 = 1/40 on each rat's offers, run over the trials with a wait time and reset at each session as in GenerateSynthData_Bayes.m (the first trial of a session has no prior and is left out). Readings: (i) the variance of the three block probabilities; (ii) the variance of the block value with blocks coded 1 low, 2 mixed, 3 high. Under (ii) a less uncertain belief has a smaller variance, the sign of the statement. Under (i) the variance is largest when one block is certain, so a less uncertain belief has a larger variance, the opposite sign: the posterior variance is then above the prior variance when the posterior is less uncertain. The entropy of the belief (h) is shown as a variant. Each is tested over trials and over per-rat means (signed-rank).

In [ ]:
rows = []
for r in rats:
    rm = M.RatModel(t[t.subject == r])
    ok = ~rm.newsess
    pr, po = rm.prior[ok], rm.post[ok]
    v_i = (np.var(pr, 1), np.var(po, 1))
    code = np.array([2.0, 3.0, 1.0])  # mixed, high, low
    def bv(p):
        m = p @ code
        return p @ code ** 2 - m ** 2
    v_ii = (bv(pr), bv(po))
    ent = lambda p: -(p * np.log(np.clip(p, 1e-300, None))).sum(1)
    v_h = (ent(pr), ent(po))
    rows.append(dict(rat=r, n=int(ok.sum()), i_prior=v_i[0].mean(), i_post=v_i[1].mean(), ii_prior=v_ii[0].mean(), ii_post=v_ii[1].mean(),
                     i_frac=float((v_i[1] > v_i[0]).mean()), ii_frac=float((v_ii[1] < v_ii[0]).mean()),
                     i_p=stats.wilcoxon(v_i[1], v_i[0], alternative="greater").pvalue,
                     ii_p=stats.wilcoxon(v_ii[1], v_ii[0], alternative="less").pvalue,
                     h_prior=v_h[0].mean(), h_post=v_h[1].mean(), h_frac=float((v_h[1] < v_h[0]).mean()),
                     h_p=stats.wilcoxon(v_h[1], v_h[0], alternative="less").pvalue))
BO = pd.DataFrame(rows).set_index("rat")
# (i) is the variance of the three probabilities: it is largest when one block is certain, so a rise
# means less uncertainty, the opposite sign from the statement's "posterior variance below prior
# variance". (ii) follows the statement's sign. The entropy (h, nats) is shown as a variant.
for rd, better in [("i", "greater"), ("ii", "less"), ("h", "less")]:
    p_rats = stats.wilcoxon(BO[rd + "_post"], BO[rd + "_prior"], alternative=better).pvalue
    p_two = stats.wilcoxon(BO[rd + "_post"], BO[rd + "_prior"]).pvalue
    print(f"reading ({rd}): mean prior {BO[rd+'_prior'].mean():.4f}, posterior {BO[rd+'_post'].mean():.4f}; "
          f"per-rat means, signed-rank two-tailed p = {p_two:.3g} (one-tailed towards less uncertainty p = {p_rats:.3g}); "
          f"over trials: per-rat test p < 0.001 in {int((BO[rd+'_p'] < 0.001).sum())} of {len(BO)} rats, median share of trials less uncertain {BO[rd+'_frac'].median():.3f}")
    R.setdefault("bayes", {})[rd] = (BO[rd + "_prior"].mean(), BO[rd + "_post"].mean(), p_two, int((BO[rd + "_p"] < 0.001).sum()))
# pooled over all trials of all rats

**Result: reproduced.** The criterion asks for a less uncertain posterior under the readings run. Under reading (ii) the variance of the block value falls from 0.104 to 0.082, as the statement says (per-rat means, signed-rank p = 1.8e-49; over trials, p < 0.001 in all 291 rats). Under reading (i) the variance of the block probabilities rises from 0.157 to 0.169 (p = 1.8e-49): the posterior is more peaked, which is less uncertain, but the statement's "posterior variance below prior variance" does not hold in this measure. The entropy falls from 0.358 to 0.283 nats (p = 1.8e-49; p < 0.001 in all 291 rats).

## pre-cue-session-link

> The rats and switch dates that the paper's Figure5.m lists mark which of the dandiset's sessions were run on the pre-initiation cue task, which nothing in the files marks.

Input (with_labels): `ratListAll` and `startDates` of Figure5.m (github.com/constantinoplelab/published at b7b68ca8c107d8ce4a061e16da214307f175c3e5, rat_behavior/Figures/Figure5.m), copied below; the code pairs date rr with rat rr, so the first 16 of its 20 dates are used. The files also hold a task argument, `center_port_cue`, which is compared with the dates.

In [ ]:
# From Figure5.m at b7b68ca8c107 (outside input, names and dates only)
PRECUE_RATS = ['M041', 'M042', 'M043', 'M044', 'M045', 'M046', 'M047', 'M048', 'G070', 'E005', 'E008',
               'G078', 'G076', 'G069', 'S042', 'S050']
PRECUE_DATES = ['2022-10-04', '2022-10-04', '2022-10-04', '2022-10-04', '2022-12-13', '2022-12-13', '2022-12-13',
                '2022-12-13', '2022-10-24', '2022-10-24', '2022-10-24', '2022-11-07', '2022-11-14', '2022-11-14',
                '2022-12-21', '2022-12-22', '2023-01-03', '2023-01-03', '2023-01-23', '2023-01-25']
SECOND_GROUP = ['M045', 'M046', 'M047', 'M048']  # Figure5.m secondGroup
start_date = dict(zip(PRECUE_RATS, pd.to_datetime(PRECUE_DATES[:16])))
ses = sessions.copy()
ses["day"] = ses.path.str.extract(r"-(\d{8})-\d{6}\.nwb$")[0].pipe(pd.to_datetime, format="%Y%m%d")
ses["after_date"] = ses.subject.map(start_date).notna() & (ses.day >= ses.subject.map(start_date))
ses["cue_flag"] = ses.center_port_cue == "True"
print("rats with any center_port_cue = True session:", sorted(ses[ses.cue_flag].subject.unique()))
print("of them in Figure5.m's list:", sorted(set(ses[ses.cue_flag].subject) & set(PRECUE_RATS)))
tab = ses[ses.subject.isin(PRECUE_RATS)].groupby("subject").apply(lambda g: pd.Series({
    "sessions": len(g), "before_date": int((~g.after_date).sum()), "after_date": int(g.after_date.sum()),
    "flag_true": int(g.cue_flag.sum()), "after_and_flag": int((g.after_date & g.cue_flag).sum()),
    "first_flag_day": g.loc[g.cue_flag, "day"].min(), "date": start_date[g.name]}))
print(tab.to_string())
t["after_date"] = ses.after_date.to_numpy()[t.sess.to_numpy()]
t["cue_flag"] = ses.cue_flag.to_numpy()[t.sess.to_numpy()]
# post-violation trials: the previous trial in the session was a violation
pv = t.is_violation.shift(1).where(t.trial > 0).fillna(False).astype(bool)
t["post_violation"] = pv.to_numpy()
rows = []
for r in PRECUE_RATS:
    g = t[(t.subject == r) & (t.block == 1) & t.z_iti.notna() & ~t.post_violation]
    out = {"rat": r}
    for nm, m in [("before", ~g.after_date), ("after", g.after_date)]:
        h = g[m]
        if len(h) < 50:
            out[nm + "_slope"], out[nm + "_anova_p"] = np.nan, np.nan
            continue
        out[nm + "_slope"] = np.polyfit(np.log2(h.vol.astype(float)), h.z_iti, 1)[0]
        out[nm + "_anova_p"] = stats.f_oneway(*[h.z_iti[h.vol == v] for v in [5, 10, 20, 40, 80]]).pvalue
    b = out["before_slope"]
    out["linked"] = bool(out["after_anova_p"] < 0.05 and (out["after_slope"] < b if np.isfinite(b) else out["after_slope"] < 0))
    rows.append(out)
LINK = pd.DataFrame(rows).set_index("rat")
print(LINK.round(4).to_string())
print("linked rats:", int(LINK.linked.sum()), "of 16")
R["link"] = int(LINK.linked.sum())
# the same test with the center_port_cue flag in place of the dates
rows = []
for r in PRECUE_RATS:
    g = t[(t.subject == r) & (t.block == 1) & t.z_iti.notna() & ~t.post_violation]
    s_ = {nm: (np.polyfit(np.log2(g[m].vol.astype(float)), g[m].z_iti, 1)[0] if m.sum() > 50 else np.nan)
          for nm, m in [("flag_false_or_absent", ~g.cue_flag), ("flag_true", g.cue_flag)]}
    rows.append({"rat": r, **s_})
print("slope of z-scored initiation time on log2 offer, by center_port_cue:\n", pd.DataFrame(rows).set_index("rat").round(4).to_string())
# Variants (not the claims file's reading; see the question to a reader): (1) the link test with the
# flagged sessions in place of the sessions after the date; (2) S042 and S050 paired with the last
# two of Figure5.m's 20 dates (2023-01-23 and 2023-01-25), near their first flagged days.
def link_test(g, after):
    out = {}
    for nm, m in [("before", ~after), ("after", after)]:
        h = g[m]
        if len(h) < 50:
            out[nm + "_slope"], out[nm + "_anova_p"] = np.nan, np.nan
            continue
        out[nm + "_slope"] = np.polyfit(np.log2(h.vol.astype(float)), h.z_iti, 1)[0]
        out[nm + "_anova_p"] = stats.f_oneway(*[h.z_iti[h.vol == v] for v in [5, 10, 20, 40, 80]]).pvalue
    b = out["before_slope"]
    out["linked"] = bool(out["after_anova_p"] < 0.05 and (out["after_slope"] < b if np.isfinite(b) else out["after_slope"] < 0))
    return out
LINK_FLAG = pd.DataFrame({r: link_test(g, g.cue_flag) for r in PRECUE_RATS
                          for g in [t[(t.subject == r) & (t.block == 1) & t.z_iti.notna() & ~t.post_violation]]}).T
print("(1) flagged sessions in place of the dates:\n", LINK_FLAG.to_string())
print("    linked rats:", int(LINK_FLAG.linked.sum()), "of 16")
alt = {"S042": pd.Timestamp(PRECUE_DATES[18]), "S050": pd.Timestamp(PRECUE_DATES[19])}
for r, dt in alt.items():
    g = t[(t.subject == r) & (t.block == 1) & t.z_iti.notna() & ~t.post_violation]
    day = ses.day.to_numpy()[g.sess.to_numpy()]
    o = link_test(g, pd.Series(day >= np.datetime64(dt), index=g.index))
    print(f"(2) {r} with date {dt.date()}: " + ", ".join(f"{k} {v:.4g}" if isinstance(v, float) else f"{k} {v}" for k, v in o.items()))
R["link_variants"] = int(LINK_FLAG.linked.sum())

**Result: partially reproduced.** 14 of the 16 rats are linked by the test; S042 and S050 are not (ANOVA p = 0.151 and 0.597 after the date). The files mark the pre-cue sessions themselves: the task argument `center_port_cue` is True only in sessions of these 16 rats, and on those sessions every one of them has a negative slope of initiation time on offer (-0.0084 to -0.0690), against -0.0076 to +0.0043 where it is False or absent. Some sessions after Figure5.m's dates are not flagged (S042 13 of 25, S050 6 of 11, E005 15 of 21). Two variants, not the claims file's reading (a question to a reader asks about both): with the flagged sessions in place of the dates, 15 of 16 rats are linked (S042 p = 0.014, S050 p = 0.45); pairing S042 and S050 with Figure5.m's last two dates (2023-01-23 and 2023-01-25), near their first flagged days (2023-01-24 and 2023-02-01), links S042 (p = 0.0085) and not S050 (p = 0.59).

## fig5b-pre-cue-initiation-sensitive-to-offer

> With the reward cue played before trial initiation, trial initiation times become sensitive to the offered reward.

Pre-cue sessions from the link (dates of Figure5.m): for the 12 rats that switched, (a) sessions after the first 30 pre-cue sessions (Methods) or (b) all sessions from the date (Figure5.m); for the 4 of `secondGroup`, all sessions; original-task sessions are those before the date. As variants, the same with the sessions the files flag (`center_port_cue` True): (a_flag) after the first 30 flagged sessions, and (flag) all flagged sessions. The dandiset holds only included sessions, so the paper's "no session exclusion" cannot be applied, and the 30 sessions are counted over the deposited ones. Per rat, mean z-scored initiation time by offer in mixed blocks, post-violation trials left out, one-way ANOVA over offers.

In [ ]:
pc_rank = ses[ses.after_date].groupby("subject").cumcount()
ses["precue_n"] = np.nan
ses.loc[pc_rank.index, "precue_n"] = pc_rank
t["precue_n"] = ses.precue_n.to_numpy()[t.sess.to_numpy()]
flag_rank = ses[ses.cue_flag].groupby("subject").cumcount()
ses["flag_n"] = np.nan
ses.loc[flag_rank.index, "flag_n"] = flag_rank
t["flag_n"] = ses.flag_n.to_numpy()[t.sess.to_numpy()]
def precue_mask(reading):
    """a: after 30 sessions from the date (Methods); b: from the date (Figure5.m); a_flag and flag:
    the same with the sessions the files flag (center_port_cue True), variants."""
    if reading in ("a", "b"):
        m = t.after_date.copy()
        if reading == "a":
            m &= (t.precue_n >= 30) | t.subject.isin(SECOND_GROUP)
    else:
        m = t.cue_flag.copy()
        if reading == "a_flag":
            m &= (t.flag_n >= 30) | t.subject.isin(SECOND_GROUP)
    return m
FIG5 = {}
for reading in ["a", "b", "a_flag", "flag"]:
    pm = precue_mask(reading)
    res = {}
    for task, m in [("pre-cue", pm), ("original", ~t.after_date & t.subject.isin(PRECUE_RATS))]:
        g = t[m & t.subject.isin(PRECUE_RATS) & (t.block == 1) & t.z_iti.notna() & ~t.post_violation]
        cur = g.groupby(["subject", "vol"]).z_iti.mean().unstack()
        pa = g.groupby("subject").apply(lambda h: stats.f_oneway(*[h.z_iti[h.vol == v] for v in [5, 10, 20, 40, 80] if (h.vol == v).sum() > 1]).pvalue)
        res[task] = (cur, pa)
        print(f"[{reading}] {task}: rats {len(cur)}, population mean z by offer {cur.mean().round(3).to_dict()}; "
              f"decreasing at every step: {bool((np.diff(cur.mean().to_numpy()) < 0).all())}; rats ANOVA p < 0.05: {int((pa < 0.05).sum())}")
    FIG5[reading] = res
    R.setdefault("fig5b", {})[reading] = (bool((np.diff(res["pre-cue"][0].mean().to_numpy()) < 0).all()),
                                          int((res["pre-cue"][1] < 0.05).sum()), int((res["original"][1] < 0.05).sum()),
                                          len(res["original"][0]))
print("trials per rat in reading (a):", t[precue_mask("a") & t.subject.isin(PRECUE_RATS)].groupby("subject").size().to_dict())

**Result: partially reproduced.** From Figure5.m's dates on (reading b), the population mean z-scored initiation time falls at every step from 5 to 80 uL (-0.041 to -0.178), and 14 of 16 rats reach ANOVA p < 0.05, against 0 of 12 on the original task; the flagged sessions give the same (-0.037 to -0.178, 15 of 16 rats). After 30 pre-cue sessions (the Methods' reading a; 11 rats have such sessions) all 11 reach p < 0.05 and the mean falls from 10 to 80 uL (0.024 to -0.165), but it rises from 5 to 10 uL (0.016 to 0.024); counting 30 flagged sessions gives the same (0.013 to 0.023). The criterion holds under the code's reading and in part under the Methods'.

## fig5c-pre-cue-initiation-still-sensitive-to-previous-reward

> On the pre-initiation cue task, trial initiation times for 20 uL in mixed blocks are still modulated by the previous reward, for 13 of 16 rats and for the population.

The test of fig2k under each of its readings, on the pre-cue sessions of fig5b (readings a and b, and the flagged sessions as a variant). The code's trials without post-violation trials are shown as in fig2k.

In [ ]:
for reading in ["a", "b", "flag"]:
    pm = precue_mask(reading) & t.subject.isin(PRECUE_RATS)
    for name, m in [("M: 20 uL trials", t.vol == 20), ("C: all mixed trials", pd.Series(True, index=t.index)),
                    ("C without post-violation trials", ~t.post_violation)]:
        d = t.loc[pm & m & (t.block == 1) & t.z_iti.notna() & t.prev_vol.notna(), ["subject", "z_iti", "prev_vol"]]
        pk = d.groupby("subject").apply(lambda g: M.ranksum_p(g.z_iti[g.prev_vol < 20], g.z_iti[g.prev_vol > 20]))
        mk = pd.DataFrame({"below": d[d.prev_vol < 20].groupby("subject").z_iti.mean(), "above": d[d.prev_vol > 20].groupby("subject").z_iti.mean()})
        pp = M.ranksum_p(mk.below, mk.above)
        n_dir = int(((pk < 0.05) & (mk.below > mk.above)).sum())
        print(f"[sessions {reading}, {name}] rats p < 0.05: {int((pk < 0.05).sum())} of {len(pk)} (expected 13 +- 1), slower after small offers {n_dir}; "
              f"population mean below {mk.below.mean():+.3f}, above {mk.above.mean():+.3f}; rank-sum p = {pp:.3g}; signed-rank p = {stats.wilcoxon(mk.below, mk.above).pvalue:.3g}")
        R.setdefault("fig5c", {})[(reading, name)] = (int((pk < 0.05).sum()), pp, mk.below.mean() > mk.above.mean())

**Result: partially reproduced.** With sessions from the dates and all mixed block trials (the code's readings), 12 of 16 rats have p < 0.05 (expected 13 +- 1; 11 slower after small offers), and the population is slower after small offers (+0.065 against -0.128 z; rank-sum p = 4.7e-06); the flagged sessions give the same (12 of 16, p = 3.9e-06). The Methods' 20 uL reading gives 15 of 16 rats but a population that is faster after small offers (rank-sum p = 0.337), and the code's trials without post-violation trials give 15 of 16 rats, 4 of them slower after small offers, and a population faster after small offers (p = 0.283). After 30 pre-cue sessions only 11 rats have data, 9 of them significant with all mixed trials (rank-sum p = 6.4e-04). As in fig2k, the count and direction hold only with the post-violation trials.

## fig5d-pre-cue-initiation-depends-on-previous-block

> On the pre-initiation cue task, how quickly rats initiate trials in mixed blocks still depends on the previous block.

Initiation time transition curves as in fig2a (z-scored, detrended per session), on the pre-cue sessions; mean over trials 1 to 20 of mixed blocks after high and after low blocks, signed-rank across the 16 rats.

In [ ]:
for reading in ["a", "b", "flag"]:
    pm = precue_mask(reading) & t.subject.isin(PRECUE_RATS)
    tr5 = trans[pm.to_numpy()[trans.idx.to_numpy()]]
    c5 = M.transition_curves(t, t.iti_dt_sess, tr5, rats=PRECUE_RATS)
    ab = pd.DataFrame({'h': c5[HM].loc[:, 1:20].mean(1), 'l': c5[LM].loc[:, 1:20].mean(1)}).dropna()
    a, b_ = ab.h, ab.l
    p = stats.wilcoxon(a, b_).pvalue
    print(f"[sessions {reading}] trials 1-20 of mixed blocks: after high {a.mean():+.3f}, after low {b_.mean():+.3f}; rats higher after high {int((a > b_).sum())} of {len(a)}; signed-rank p = {p:.3g}")
    R.setdefault("fig5d", {})[reading] = (a.mean(), b_.mean(), p)

**Result: reproduced.** With sessions from the dates (reading b), initiation over trials 1-20 of mixed blocks is higher after high blocks (+0.136) than after low blocks (-0.114) in all 16 rats, signed-rank p = 3.1e-05; after 30 pre-cue sessions (reading a) in all 11 rats with data, p = 9.8e-04; with the flagged sessions in all 16 rats, p = 3.1e-05.

In [ ]:
# free memory before reading the deposit
for c in ["z_wt", "wt_raw", "iti_dt_pool", "prev_block_of_block", "after_date", "cue_flag", "post_violation", "precue_n"]:
    if c in t:
        del t[c]
import gc; gc.collect()
print("peak memory so far (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

## Figure 4: data from the paper's Zenodo deposit

Figure 4 uses all final-stage sessions, including those that fail the inclusion criterion (Methods, Learning dynamics). The dandiset holds only the included sessions (data check), so the claims of Figure 4 are tested on `A_Structs_all` of the paper's own deposit, doi:10.5281/zenodo.10031483 (`published.zip`, version v1), which its README describes as all sessions that ended on the final training stage, "only used for Figure 4". Only trials with `trainingstage` 9 (the paper's stage 8) are kept. Wait times above the rat's `wait_thresh` are removed and initiation times above the rat's 99th percentile (macro_regress_overtraining_V2.m). The deposit's session dates are Matlab datetime objects that this reader cannot decode, so sessions are taken in file order; the cell checks that order by finding each dandiset session in the deposit from the initiation times of its trials 1 to 5 and comparing the order of the matches with the dandiset's dates. A match also shows that the dandiset's `inter_trial_interval` of a trial is the deposit's `ITI` of the same trial.

In [ ]:
TZ = time.time()
zz = M.zenodo_zip()
z_rats, z_sex = M.zenodo_rat_list(zz)
Z = M.load_zenodo_astructs(zz, z_rats)
print(f"loaded {len(Z)} trials of {Z.subject.nunique()} rats in {time.time()-TZ:.0f} s; trainingstage counts:", Z.trainingstage.value_counts().to_dict())
print("rats in the deposit's ratList that are in the dandiset:", len(set(z_rats) & set(rats)), "of", len(z_rats))
# Order and alignment check. The deposit's `date` field is a Matlab datetime object that scipy cannot
# decode, so each dandiset session is looked up in the deposit by the initiation times of its trials
# 1 to 5 (rounded to 0.01 s): a match means the dandiset's inter_trial_interval of trial i is the
# deposit's ITI of trial i, and the matched deposit sessions should come in the dandiset's date order.
def first_itis(df, sess, iti):
    d_ = df.loc[df.trial.between(1, 5), ["subject", sess, iti]]
    return d_.groupby(["subject", sess])[iti].apply(lambda x: tuple(np.round(x.to_numpy(), 2)))
Z9 = Z.loc[Z.trainingstage == 9, ["subject", "sess_in_rat", "ITI"]]
Z9 = Z9.assign(trial=Z9.groupby(["subject", "sess_in_rat"]).cumcount())
zsig, dsig = first_itis(Z9, "sess_in_rat", "ITI"), first_itis(t, "sess", "inter_trial_interval")
n_d = n_m = n_amb = 0
out_of_order, order_detail = [], {}
for r in rats:
    lut = {}
    for k, v in zsig.loc[r].items():
        lut.setdefault(v, []).append(k)
    pos = [lut.get(v, []) for v in dsig.loc[r].to_numpy()]
    n_d += len(pos); n_m += sum(len(p) > 0 for p in pos); n_amb += sum(len(p) > 1 for p in pos)
    first = [p[0] for p in pos if len(p) == 1]
    if np.any(np.diff(first) <= 0):
        out_of_order.append(r)
        order_detail[r] = (len(first), int((np.diff(first) <= 0).sum()))
print(f"dandiset sessions found in the deposit by their initiation times: {n_m} of {n_d} ({n_amb} with more than one match); "
      f"rats whose matched deposit sessions are not in the dandiset's date order: {len(out_of_order)} {out_of_order[:10]}")
print("   out-of-order rats (matched sessions, backward steps):", order_detail)
del Z9, zsig, dsig
Z = Z.loc[Z.trainingstage == 9, ["subject", "sess_in_rat", "catch", "optout", "reward", "block", "hits",
                                  "wait_time", "wait_thresh", "ITI"]].reset_index(drop=True)
bad = Z.reward.isna() | Z.block.isna() | ~Z.reward.round().isin(list(M.VOL_MAP))
print("final-stage trials without a reward, block, or with an offer outside the ten volumes (dropped):", int(bad.sum()))
Z = Z[~bad].reset_index(drop=True)
Z["sess"] = Z.groupby(["subject", "sess_in_rat"], sort=False).ngroup()
Z["sess_in_rat"] = Z.groupby("subject").sess.transform(lambda s: s.rank(method="dense").astype(int) - 1)
Z["trial"] = Z.groupby("sess").cumcount()
Z["vol"] = Z.reward.round().astype(int).map(M.VOL_MAP)
Z["block"] = Z.block.astype(int)
Z["wt"] = Z.wait_time.where((Z.catch == 1) & (Z.optout == 1) & (Z.wait_time <= Z.wait_thresh))
Z = Z.drop(columns=["wait_time", "wait_thresh", "catch", "optout", "reward"])
nz = Z.groupby("subject").sess_in_rat.max() + 1
nd = sessions.groupby("subject").size()
print(f"final-stage sessions per rat: deposit median {nz.median():.0f} (range {nz.min()}-{nz.max()}), dandiset median {nd.median():.0f}; "
      f"rats with more sessions in the deposit than the dandiset: {int((nz.reindex(nd.index) > nd).sum())}, fewer: {int((nz.reindex(nd.index) < nd).sum())}")
print(f"final-stage trials: deposit {len(Z)}, dandiset {len(t)}")

def prep_iti(df):
    """Initiation times as block_dynamics_latency.m: first trial of each session NaN, values above
    the rat's 99th percentile (over the trials passed) removed, z-scored per rat."""
    v = df.ITI.where(df.trial > 0)
    p99 = v.groupby(df.subject).transform(lambda x: np.nanpercentile(x, 99) if x.notna().any() else np.nan)
    v = v.where(v <= p99)
    g = v.groupby(df.subject)
    return v, (v - g.transform("mean")) / g.transform("std")

Z["iti"], Z["z_iti"] = prep_iti(Z)

## fig4a-wait-time-block-sensitivity-emerges

> When the hidden blocks are introduced, wait times for 20 uL become sensitive to the block only gradually over training, while trial initiation times are block sensitive from the first session.

First 50 final-stage sessions per rat, in pairs; per pair the wait time ratio (20 uL, high over low) and the initiation time ratio (high over low). Per session, wait time on opt-out catch trials regressed on current reward and block (1 low, 2 mixed, 3 high); the block coefficient smoothed with a 5-session moving average, outliers (3 scaled MADs from a 5-point moving median) set to NaN, then averaged over rats (Methods, Learning dynamics). The reward enters as log2 offer (the Methods say "current reward"; linear offer is shown as a variant).

In [ ]:
Z50 = Z[Z.sess_in_rat < 50].copy()
Z50["pair"] = Z50.sess_in_rat // 2
w = Z50[(Z50.vol == 20) & Z50.wt.notna() & Z50.block.isin([2, 3])]
wr = w.groupby(["subject", "pair", "block"]).wt.mean().unstack()
wt_pair = (wr[2] / wr[3]).unstack()
i_ = Z50[Z50.iti.notna() & Z50.block.isin([2, 3])]
ir = i_.groupby(["subject", "pair", "block"]).iti.mean().unstack()
iti_pair = (ir[2] / ir[3]).unstack()
print("rats with a ratio in pair 1: wait", int(wt_pair[0].notna().sum()), "initiation", int(iti_pair[0].notna().sum()))
print("mean wait time ratio by pair:", wt_pair.mean().round(3).to_list())
print("mean initiation ratio by pair:", iti_pair.mean().round(3).to_list())
f_iti = iti_pair[0].dropna()
p1 = stats.wilcoxon(f_iti - 1).pvalue
w_first, w_last = wt_pair[0].mean(), wt_pair.loc[:, 15:24].mean(1).mean()
print(f"(1) first pair initiation ratio: mean {f_iti.mean():.3f}, n {len(f_iti)}, signed-rank against 1 p = {p1:.3g}")
print(f"(2) wait time ratio first pair {w_first:.3f} (|1 - r| = {abs(1-w_first):.3f}); mean over pairs 16-25 {w_last:.3f} (|1 - r| = {abs(1-w_last):.3f})")
# the same comparison with the median over rats and with the ratio of trial means pooled over rats
pooled = w.groupby(["pair", "block"]).wt.mean().unstack()
pooled = pooled[2] / pooled[3]
med_last = wt_pair.loc[:, 15:24].median(1).median()
print(f"    median over rats: first pair {wt_pair[0].median():.3f}, pairs 16-25 {med_last:.3f}; "
      f"pooled trials: first pair {pooled.loc[0]:.3f}, pairs 16-25 {pooled.loc[15:24].mean():.3f}")
nct = w[w.pair == 0].groupby(["subject", "block"]).size().unstack()
print(f"    20 uL opt-out catch trials per rat in pair 1: high median {nct[2].median():.0f}, low median {nct[3].median():.0f}; "
      f"rats with a ratio {int(wt_pair[0].notna().sum())}; first pair ratio range {wt_pair[0].min():.2f}-{wt_pair[0].max():.2f}")
keep = ~wt_pair.index.isin(out_of_order)
print(f"    without the {len(out_of_order)} rats whose deposit order differs from the dates: mean first pair {wt_pair[0][keep].mean():.3f}, "
      f"pairs 16-25 {wt_pair.loc[keep, 15:24].mean(axis=1).mean():.3f}; initiation first pair {iti_pair[0][~iti_pair.index.isin(out_of_order)].mean():.3f}")
R["fig4a_var"] = (wt_pair[0].median(), med_last, pooled.loc[0], pooled.loc[15:24].mean())

def session_coefs(df, kind, logrew=True):
    out = {}
    for (r, s), g in df.groupby(["subject", "sess_in_rat"]):
        if kind == "block":
            h = g[g.wt.notna()]
            if len(h) < 5 or h.block.nunique() < 2:
                continue
            x1 = np.log2(h.vol.astype(float)) if logrew else h.vol.astype(float)
            blk = h.block.map({3: 1, 1: 2, 2: 3}).astype(float)
            X = np.c_[x1, blk, np.ones(len(h))]
            out[(r, s)] = np.linalg.lstsq(X, h.wt.to_numpy(), rcond=None)[0][1]
        else:
            pr = np.log2(g.vol.astype(float).shift(1)) * g.hits.shift(1)
            ok = pr.notna() & g.iti.notna()
            if ok.sum() < 10:
                continue
            X = np.c_[pr[ok], np.ones(ok.sum())]
            out[(r, s)] = np.linalg.lstsq(X, g.iti[ok].to_numpy(), rcond=None)[0][0]
    return pd.Series(out).unstack()

def smooth_clean(C):
    """5-session moving average per rat, then outliers (3 scaled MADs from a 5-point moving median,
    Matlab isoutlier 'movmedian') set to NaN."""
    S = C.T.rolling(5, center=True, min_periods=1).mean()
    med = S.rolling(5, center=True, min_periods=1).median()
    mad = (S - med).abs().rolling(5, center=True, min_periods=1).median() * 1.4826
    S = S.where(~((S - med).abs() > 3 * mad))
    return S.T

B = {lg: smooth_clean(session_coefs(Z50, "block", lg)) for lg in [True, False]}
for lg in [True, False]:
    b = B[lg].mean()
    e, l = b.loc[0:9].mean(), b.loc[40:49].mean()
    print(f"(3) block coefficient ({'log2' if lg else 'linear'} reward): sessions 1-10 {e:.4f}, sessions 41-50 {l:.4f}; grows in size: {abs(l) > abs(e)}")
R["fig4a"] = (f_iti.mean(), p1, w_first, w_last, B[True].mean().loc[0:9].mean(), B[True].mean().loc[40:49].mean())
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(np.arange(1, 26) * 2, wt_pair.mean().values, label="wait time"); ax[0].plot(np.arange(1, 26) * 2, iti_pair.mean().values, label="initiation")
ax[0].axhline(1, color="k", lw=0.5); ax[0].legend(); ax[0].set_xlabel("session"); ax[0].set_ylabel("ratio high/low")
ax[1].plot(B[True].mean().values); ax[1].set_title("block coefficient"); plt.show()
print("plotted ranges: wait ratio", wt_pair.mean().min().round(3), "-", wt_pair.mean().max().round(3),
      "| initiation ratio", iti_pair.mean().min().round(3), "-", iti_pair.mean().max().round(3))

**Result: partially reproduced.** Initiation times are block sensitive from the first session pair (ratio 0.802 over 273 rats, signed-rank against 1 p = 5.8e-16), and the wait time block coefficient grows in size from -0.432 over sessions 1-10 to -0.706 over sessions 41-50 (linear offer: -0.393 to -0.633). The mean wait time ratio of the first pair is 1.187, further from 1 than its mean over pairs 16-25 (0.936), so part (2) of the criterion fails as written. That mean rests on a median of 5 high and 4 low block catch trials per rat, and the per-rat ratios range from 0.11 to 19.60: the median over rats (0.953 against 0.884) and the ratio of pooled trial means (0.971 against 0.886) put the first pair closer to 1. 21,302 of the 23,033 dandiset sessions are found in the deposit, with no session matched twice; in 281 rats the matches follow the dandiset's dates, and in 10 rats 1 or 2 matched sessions step back. Without those 10 rats the first pair's mean ratio is 1.200.

## fig4c-initiation-previous-reward-stable

> The effect of the previous reward on trial initiation times is comparable early and late in training.

Per session, initiation time regressed on the previous reward (log2 offer times hit, 0 when unrewarded, as fig2c-d), smoothed and cleaned as in fig4a, averaged over rats.

In [ ]:
Cp = smooth_clean(session_coefs(Z50, "prev"))
c = Cp.mean()
ce, cl = c.loc[0:9].mean(), c.loc[40:49].mean()
b = B[True].mean(); be, bl = b.loc[0:9].mean(), b.loc[40:49].mean()
rel_c, rel_b = abs(ce - cl) / abs(cl), abs(be - bl) / abs(bl)
print(f"previous reward coefficient: sessions 1-10 {ce:.4f}, 41-50 {cl:.4f}; same sign: {np.sign(ce) == np.sign(cl)}; relative change {rel_c:.3f}")
print(f"wait time block coefficient relative change {rel_b:.3f}; initiation change smaller: {rel_c < rel_b}")
print("rats with a coefficient in session 1:", int(Cp[0].notna().sum()), "and session 50:", int(Cp[49].notna().sum()) if 49 in Cp else 0)
R["fig4c"] = (ce, cl, rel_c, rel_b)

**Result: reproduced.** The previous reward coefficient is -0.588 over sessions 1-10 and -0.469 over sessions 41-50, the same sign, a relative change of 0.252 against 0.389 for the wait time block coefficient.

## fig4d-overshoot-grows-with-training

> The overshoot of trial initiation times after transitions from high to mixed blocks becomes more prominent with training, on a timescale like that of block sensitivity for wait times.

Per rat, the first 10 and the last 10 final-stage sessions (Figure4.m, nsess = 10); initiation times z-scored over those sessions and detrended with one line over trial number (block_dynamics_latency.m); high-to-mixed curves smoothed (10 trials) and averaged over rats; overshoot = maximum over trials 0 to 20 minus the value at trial 20 (Fig. 4d legend; Figure4.m). For Fig. 4e, the overshoot per chunk of 5 sessions is shown beside the wait time ratio.

In [ ]:
def overshoot_curve(df):
    df = df[["subject", "sess", "trial", "block", "ITI"]].reset_index(drop=True)
    df["iti"], df["z_iti"] = prep_iti(df)
    df["dt"] = M.detrend_iti(df, df.z_iti, per_session=False)
    tr = M.transitions(df)
    c = M.transition_curves(df, df.dt, tr)
    h = c[(2, 1)].mean()
    return float(h.loc[0:20].max() - h.loc[20]), h

nmax = Z.groupby("subject").sess_in_rat.transform("max")
early = Z[Z.sess_in_rat < 10]
late = Z[Z.sess_in_rat > nmax - 10]
oe, he = overshoot_curve(early)
ol, hl_ = overshoot_curve(late)
print(f"overshoot first 10 sessions {oe:.4f}, last 10 sessions {ol:.4f}; larger late: {ol > oe}")
print("high->mixed curve, early, trials 0..20:", he.loc[0:20].round(3).to_list())
print("high->mixed curve, late, trials 0..20:", hl_.loc[0:20].round(3).to_list())
R["fig4d"] = (oe, ol)
chunk = Z.sess_in_rat // 5
ch = {}
for k in range(20):
    ch[k] = overshoot_curve(Z[chunk == k])[0]
ch = pd.Series(ch)
print("overshoot per 5-session chunk (first 100 sessions):", ch.round(3).to_list())
print("10-point moving average:", ch.rolling(10, min_periods=1).mean().round(3).to_list())

**Result: reproduced.** The overshoot after high-to-mixed transitions is 0.025 z in the first 10 and 0.046 z in the last 10 final-stage sessions. Per 5-session chunk it goes from 0.008 to about 0.06 over the first 100 sessions (smoothed, 0.008 to 0.055).

## Summary

| Claim | Outcome |
|---|---|
| cohort-size | reproduced |
| fig1e-wait-time-increases-with-reward | reproduced |
| fig1f-wait-time-block-effect | reproduced |
| fig1i-initiation-block-effect | reproduced |
| suppfig2-initiation-block-effect-after-unrewarded | reproduced |
| suppfig3-wait-time-not-affected-by-previous-outcome | reproduced |
| fig2a-wait-time-converges-in-mixed-blocks | reproduced |
| suppfig5-mixed-to-adaptation-transitions | reproduced |
| fig2c-d-regression-on-previous-rewards | reproduced |
| fig2e-tau-differ | reproduced |
| fig2f-tau-uncorrelated | reproduced |
| fig2g-h-fast-slow-tau-groups | reproduced |
| fig2j-wait-time-not-sensitive-to-previous-offer | reproduced |
| fig2k-initiation-sensitive-to-previous-offer | partially reproduced |
| fig2l-sensitivity-greater-for-initiation | partially reproduced |
| suppfig6-results-without-detrending | reproduced |
| fig3c-wait-time-by-catch-probability | reproduced |
| bayes-observer-offer-reduces-uncertainty | reproduced |
| fig4a-wait-time-block-sensitivity-emerges | partially reproduced |
| fig4c-initiation-previous-reward-stable | reproduced |
| fig4d-overshoot-grows-with-training | reproduced |
| fig5b-pre-cue-initiation-sensitive-to-offer | partially reproduced |
| fig5c-pre-cue-initiation-still-sensitive-to-previous-reward | partially reproduced |
| fig5d-pre-cue-initiation-depends-on-previous-block | reproduced |
| suppfig12-no-sex-differences | reproduced |
| suppfig14-contextual-effects-without-threshold | reproduced |
| pre-cue-session-link | partially reproduced |

The model claims are in `models_mah_2023.ipynb`.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's:
- The dandiset holds only the sessions that pass the inclusion criterion; they are used as given. Figure 4 is tested on the paper's Zenodo deposit, which holds all final-stage sessions.
- Lags and previous trials are taken within a session; the paper's code concatenates sessions.
- Exponential fits are global grid searches in place of fmincon from random starts.
- Figure 4 takes the deposit's sessions in file order; 21,302 of the 23,033 dandiset sessions are found in it, in date order for 281 of 291 rats.
- For fig2k, fig2l, fig5b and fig5c the Methods and the code describe different trials or sessions; every reading is shown. For fig2k, fig2l and fig5c the code's result comes from the trials after violations, on which the offer repeats; without them the direction reverses, as under the Methods' reading.
- Figure 5's pre-cue sessions come from the rat names and dates of Figure5.m; the files also flag them (`center_port_cue`).

What was found about the dataset:
- Counts match the paper exactly: 291 rats (184 male, 107 female), median 33,493 trials and 71 sessions per rat.
- Only included sessions are deposited (the paper's deposit has a median of 95 final-stage sessions per rat, the dandiset 71); nothing in the dandiset says so.
- `wait_time_threshold` is the mean + 1 SD of all catch trials with a wait time, timed-out catch trials (100 s) included, where the description says catch-trial wait times.
- The task argument `center_port_cue`, which has no description, marks the pre-initiation cue sessions; it is True only for the 16 rats of Figure 5.
- Catch probability 0.3, which the paper does not list, occurs for 8 rats; two transitions go directly between high and low blocks.